In [ ]:
"""
Signaling Game with Occasionally Observed World States
(Numba-optimized version)

Requires: pip install numba

=============================================================================
CONFIGURATION - EDIT THESE PARAMETERS
=============================================================================
"""

# ---- Game parameters ----
N = 4                  # Number of states/signals/actions
P_OBSERVE = 1        # Probability receiver observes state
TIMESTEPS = 10_000_000  # Rounds per simulation
# TIMESTEPS = 100_000_000  # Rounds per simulation
N_RUNS = 1_00           # Number of simulations to run

# ---- Variant selection ----
# Action selection: "LP" (linear probability pooling), 
#                   "BP" (ball pooling), 
#                   "SO" (state only)
# ACTION_SELECTION = "LP"
# ACTION_SELECTION = "BP"
ACTION_SELECTION = "BP"

# Reinforcement: "no_gating" (reinforce both urns on success),
#                "gated" (only reinforce if urn recommended the action)
REINFORCEMENT = "no_gating"
# REINFORCEMENT = "gated"

# ---- What to run ----
RUN_SINGLE = True           # Run the single variant specified above
RUN_COMPARISON = False      # Run comparison of all 6 variants
RUN_DUAL_THRESHOLD = False  # Run SO, no-gating, n=4 with dual threshold analysis
SAVE_PLOTS = True           # Save plots to files
SHOW_PLOTS = False          # Display plots interactively
SHOW_FAILED_STRATEGIES = False  # Print final strategies for runs with payoff < 0.875

RUN_SCALING_TABLE = True
"""
=============================================================================
END OF CONFIGURATION
=============================================================================
"""

import numpy as np
from numba import njit
import matplotlib.pyplot as plt
from dataclasses import dataclass
from enum import IntEnum
import time
from datetime import datetime


class ActionSelectionEnum(IntEnum):
    LP = 0  # Linear probability pooling
    BP = 1  # Ball pooling
    SO = 2  # State only


class ReinforcementEnum(IntEnum):
    NO_GATING = 0
    GATED = 1


def parse_action_selection(s: str) -> ActionSelectionEnum:
    mapping = {"LP": ActionSelectionEnum.LP, 
               "BP": ActionSelectionEnum.BP, 
               "SO": ActionSelectionEnum.SO}
    return mapping[s.upper()]


def parse_reinforcement(s: str) -> ReinforcementEnum:
    mapping = {"NO_GATING": ReinforcementEnum.NO_GATING,
               "GATED": ReinforcementEnum.GATED}
    return mapping[s.upper().replace("-", "_")]


@dataclass
class GameConfig:
    """Configuration for the signaling game."""
    n: int = 2
    p_observe: float = 0.1
    action_selection: ActionSelectionEnum = ActionSelectionEnum.LP
    reinforcement: ReinforcementEnum = ReinforcementEnum.NO_GATING
    timesteps: int = 10000
    n_runs: int = 100

    def variant_name(self) -> str:
        action_names = {0: "LP", 1: "BP", 2: "SO"}
        reinf_names = {0: "no_gating", 1: "gated"}
        return f"{action_names[int(self.action_selection)]}, {reinf_names[int(self.reinforcement)]}"
    
    @classmethod
    def from_globals(cls):
        """Create config from global parameters."""
        return cls(
            n=N,
            p_observe=P_OBSERVE,
            timesteps=TIMESTEPS,
            n_runs=N_RUNS,
            action_selection=parse_action_selection(ACTION_SELECTION),
            reinforcement=parse_reinforcement(REINFORCEMENT)
        )


@njit
def choose_from_probs(probs):
    """Choose an index according to probability distribution."""
    r = np.random.random()
    cumsum = 0.0
    for i in range(len(probs)):
        cumsum += probs[i]
        if r < cumsum:
            return i
    return len(probs) - 1


@njit
def run_single_simulation(n, p_observe, action_selection, reinforcement, timesteps):
    """
    Run a single simulation and return final urn states and total payoff.
    """
    # Initialize urns
    Q_sender = np.ones((n, n))
    Q_recv_signal = np.ones((n, n))
    Q_recv_state = np.ones((n, n))
    
    total_payoff = 0
    
    for t in range(timesteps):
        # Nature chooses state uniformly
        state = np.random.randint(0, n)
        
        # Sender chooses signal
        sender_probs = Q_sender[state] / Q_sender[state].sum()
        signal = choose_from_probs(sender_probs)
        
        # Does receiver observe state?
        observe_state = np.random.random() < p_observe
        
        if observe_state:
            # Receiver observes both state and signal
            probs_state = Q_recv_state[state] / Q_recv_state[state].sum()
            probs_signal = Q_recv_signal[signal] / Q_recv_signal[signal].sum()
            
            # Draw recommendations for potential gated reinforcement
            rec_state = choose_from_probs(probs_state)
            rec_signal = choose_from_probs(probs_signal)
            
            # Choose action based on action selection rule
            if action_selection == 2:  # SO: state-only
                action = choose_from_probs(probs_state)
            elif action_selection == 0:  # LP: linear probability pooling
                pooled_probs = 0.5 * probs_state + 0.5 * probs_signal
                action = choose_from_probs(pooled_probs)
            else:  # BP: ball pooling
                pooled_Q = Q_recv_state[state] + Q_recv_signal[signal]
                pooled_probs = pooled_Q / pooled_Q.sum()
                action = choose_from_probs(pooled_probs)
            
            # Check success
            success = (action == state)
            if success:
                total_payoff += 1
                # Update sender
                Q_sender[state, signal] += 1
                
                # Update receiver based on reinforcement rule
                if reinforcement == 0:  # NO_GATING
                    Q_recv_state[state, action] += 1
                    Q_recv_signal[signal, action] += 1
                else:  # GATED
                    if rec_state == action:
                        Q_recv_state[state, action] += 1
                    if rec_signal == action:
                        Q_recv_signal[signal, action] += 1
        else:
            # Receiver only observes signal
            probs_signal = Q_recv_signal[signal] / Q_recv_signal[signal].sum()
            action = choose_from_probs(probs_signal)
            
            success = (action == state)
            if success:
                total_payoff += 1
                Q_sender[state, signal] += 1
                Q_recv_signal[signal, action] += 1
    
    return Q_sender, Q_recv_signal, Q_recv_state, total_payoff


@njit
def compute_expected_payoff(n, p_observe, action_selection,
                            Q_sender, Q_recv_signal, Q_recv_state):
    """Compute expected payoff given current strategies."""
    # Sender strategy
    sender_strat = np.zeros((n, n))
    for i in range(n):
        sender_strat[i] = Q_sender[i] / Q_sender[i].sum()
    
    # Receiver strategies
    recv_signal_strat = np.zeros((n, n))
    recv_state_strat = np.zeros((n, n))
    for i in range(n):
        recv_signal_strat[i] = Q_recv_signal[i] / Q_recv_signal[i].sum()
        recv_state_strat[i] = Q_recv_state[i] / Q_recv_state[i].sum()
    
    # Expected payoff for signal-only rounds
    exp_signal_only = 0.0
    for i in range(n):
        for j in range(n):
            exp_signal_only += sender_strat[i, j] * recv_signal_strat[j, i]
    exp_signal_only /= n
    
    # Expected payoff for state-observed rounds
    exp_state_obs = 0.0
    for i in range(n):
        for j in range(n):
            p_signal = sender_strat[i, j]
            
            if action_selection == 2:  # SO
                p_correct = recv_state_strat[i, i]
            elif action_selection == 0:  # LP
                p_correct = 0.5 * recv_state_strat[i, i] + 0.5 * recv_signal_strat[j, i]
            else:  # BP
                Q_pooled = Q_recv_state[i] + Q_recv_signal[j]
                p_correct = Q_pooled[i] / Q_pooled.sum()
            
            exp_state_obs += p_signal * p_correct
    exp_state_obs /= n
    
    return (1 - p_observe) * exp_signal_only + p_observe * exp_state_obs


@njit
def run_multiple_simulations(n, p_observe, action_selection, reinforcement, 
                              timesteps, n_runs):
    """Run multiple simulations and collect statistics."""
    final_expected_payoffs = np.zeros(n_runs)
    average_payoffs = np.zeros(n_runs)
    
    for run in range(n_runs):
        Q_sender, Q_recv_signal, Q_recv_state, total_payoff = \
            run_single_simulation(n, p_observe, action_selection, 
                                  reinforcement, timesteps)
        
        final_expected_payoffs[run] = compute_expected_payoff(
            n, p_observe, action_selection,
            Q_sender, Q_recv_signal, Q_recv_state
        )
        average_payoffs[run] = total_payoff / timesteps
    
    return final_expected_payoffs, average_payoffs


def run_experiments(config: GameConfig, verbose: bool = True) -> dict:
    """Run experiments with given config."""
    if verbose:
        print(f"Running {config.n_runs} simulations...", flush=True)
        start = time.time()
    
    # If we need to show failed strategies, run one at a time
    if SHOW_FAILED_STRATEGIES:
        final_expected_payoffs = []
        average_payoffs = []
        failed_strategies = []
        
        for run in range(config.n_runs):
            Q_sender, Q_recv_signal, Q_recv_state, total_payoff = run_single_simulation(
                config.n,
                config.p_observe,
                int(config.action_selection),
                int(config.reinforcement),
                config.timesteps
            )
            
            exp_payoff = compute_expected_payoff(
                config.n, config.p_observe, int(config.action_selection),
                Q_sender, Q_recv_signal, Q_recv_state
            )
            avg_payoff = total_payoff / config.timesteps
            
            final_expected_payoffs.append(exp_payoff)
            average_payoffs.append(avg_payoff)
            
            # Store strategies for failed runs
            if exp_payoff < 0.875:
                sender_strat = Q_sender / Q_sender.sum(axis=1, keepdims=True)
                recv_signal_strat = Q_recv_signal / Q_recv_signal.sum(axis=1, keepdims=True)
                recv_state_strat = Q_recv_state / Q_recv_state.sum(axis=1, keepdims=True)
                failed_strategies.append({
                    'run': run + 1,
                    'exp_payoff': exp_payoff,
                    'sender': sender_strat,
                    'recv_signal': recv_signal_strat,
                    'recv_state': recv_state_strat
                })
            
            # Print timestamp when run finishes
            timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            print(f"  [{timestamp}] Progress: {run + 1}/{config.n_runs} runs completed", flush=True)
        
        final_expected_payoffs = np.array(final_expected_payoffs)
        average_payoffs = np.array(average_payoffs)
        
        # Print failed strategies
        if failed_strategies:
            print(f"\n{'='*60}", flush=True)
            print(f"FAILED RUNS (expected payoff < 0.875): {len(failed_strategies)}", flush=True)
            print(f"{'='*60}", flush=True)
            for fs in failed_strategies:
                print(f"\nRun {fs['run']} - Expected payoff: {fs['exp_payoff']:.4f}", flush=True)
                print(f"  Sender strategy P(signal|state):", flush=True)
                print(f"  {np.array2string(fs['sender'], precision=3, suppress_small=True)}", flush=True)
                print(f"  Receiver signal strategy P(action|signal):", flush=True)
                print(f"  {np.array2string(fs['recv_signal'], precision=3, suppress_small=True)}", flush=True)
                if config.p_observe > 0:
                    print(f"  Receiver state strategy P(action|state):", flush=True)
                    print(f"  {np.array2string(fs['recv_state'], precision=3, suppress_small=True)}", flush=True)
            print(flush=True)
    else:
        # Run in batches (faster when not tracking failed strategies)
        batch_size = max(1, config.n_runs // 100)
        all_final = []
        all_avg = []
        
        for i in range(0, config.n_runs, batch_size):
            runs_this_batch = min(batch_size, config.n_runs - i)
            final_batch, avg_batch = run_multiple_simulations(
                config.n,
                config.p_observe,
                int(config.action_selection),
                int(config.reinforcement),
                config.timesteps,
                runs_this_batch
            )
            all_final.append(final_batch)
            all_avg.append(avg_batch)
            
            if verbose:
                completed = i + runs_this_batch
                # Print timestamp when batch finishes
                timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                print(f"  [{timestamp}] Progress: {completed}/{config.n_runs} runs completed", flush=True)
        
        final_expected_payoffs = np.concatenate(all_final)
        average_payoffs = np.concatenate(all_avg)
    
    if verbose:
        elapsed = time.time() - start
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        print(f"[{timestamp}] Completed in {elapsed:.2f} seconds", flush=True)
    
    success_rate = np.mean(final_expected_payoffs >= 0.875)
    
    return {
        'final_expected_payoffs': final_expected_payoffs,
        'average_payoffs': average_payoffs,
        'success_rate': success_rate,
        'mean_final_expected': np.mean(final_expected_payoffs),
        'mean_average_payoff': np.mean(average_payoffs),
    }


def plot_payoff_distribution(results: dict, config: GameConfig,
                              save_path: str = None, show: bool = True):
    """Plot the distribution of final expected payoffs."""
    # Use a clean style with white background
    plt.style.use('default')
    
    fig, ax = plt.subplots(figsize=(10, 6), facecolor='white')
    ax.set_facecolor('white')
    
    payoffs = results['final_expected_payoffs']
    
    n_bins = min(30, len(payoffs) // 3)
    ax.hist(payoffs, bins=n_bins, density=True, alpha=0.7, color='steelblue',
            edgecolor='white', linewidth=0.5)
    
    ax.axvline(x=0.875, color='red', linestyle='--', linewidth=2,
               label='Success threshold (0.875)')
    
    mean_payoff = results['mean_final_expected']
    ax.axvline(x=mean_payoff, color='green', linestyle='-', linewidth=2,
               label=f'Mean = {mean_payoff:.3f}')
    
    ax.set_xlabel('Final Expected Payoff', fontsize=12, color='black')
    ax.set_ylabel('Density', fontsize=12, color='black')
    ax.set_title(f'Distribution of Final Expected Payoffs\n'
                 f'Variant: {config.variant_name()}, n={config.n}, '
                 f'p(observe)={config.p_observe}, T={config.timesteps}\n'
                 f'Success rate (≥0.875): {results["success_rate"]*100:.1f}%',
                 fontsize=11, color='black')
    ax.legend(loc='upper left', facecolor='white', edgecolor='black')
    ax.set_xlim(0, 1.05)
    
    
    ax.tick_params(colors='black')
    for spine in ax.spines.values():
        spine.set_color('black')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight', 
                    facecolor='white', edgecolor='none')
        print(f"Plot saved to {save_path}", flush=True)
    
    if show:
        plt.show()
    else:
        plt.close()
    
    return fig


def compare_variants(save_plots: bool = True, show_plots: bool = False):
    """Compare all six variants using global parameters N, P_OBSERVE, TIMESTEPS, N_RUNS."""
    variants = [
        (ActionSelectionEnum.LP, ReinforcementEnum.NO_GATING),
        (ActionSelectionEnum.BP, ReinforcementEnum.NO_GATING),
        (ActionSelectionEnum.LP, ReinforcementEnum.GATED),
        (ActionSelectionEnum.BP, ReinforcementEnum.GATED),
        (ActionSelectionEnum.SO, ReinforcementEnum.NO_GATING),
        (ActionSelectionEnum.SO, ReinforcementEnum.GATED),
    ]
    
    all_results = {}
    
    print(f"\n{'='*60}")
    print(f"Comparing all variants: n={N}, p(observe)={P_OBSERVE}, T={TIMESTEPS}")
    print(f"Running {N_RUNS} simulations per variant")
    print(f"{'='*60}\n")
    
    total_start = time.time()
    
    for action_sel, reinf in variants:
        config = GameConfig(
            n=N,
            p_observe=P_OBSERVE,
            action_selection=action_sel,
            reinforcement=reinf,
            timesteps=TIMESTEPS,
            n_runs=N_RUNS
        )
        
        variant_name = config.variant_name()
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        print(f"[{timestamp}] Running variant: {variant_name}")
        
        results = run_experiments(config, verbose=False)
        all_results[variant_name] = results
        
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        print(f"[{timestamp}]   Mean final expected payoff: {results['mean_final_expected']:.4f}")
        print(f"  Mean average payoff: {results['mean_average_payoff']:.4f}")
        print(f"  Success rate (≥0.875): {results['success_rate']*100:.1f}%")
        print()
        
        if save_plots:
            safe_name = variant_name.replace(", ", "_").replace(" ", "_")
            plot_payoff_distribution(results, config,
                                      save_path=f"payoff_dist_{safe_name}.png",
                                      show=show_plots)
    
    total_elapsed = time.time() - total_start
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print(f"[{timestamp}] Total time: {total_elapsed:.2f} seconds")
    
    # Summary comparison plot
    if save_plots:
        plt.style.use('default')
        fig, axes = plt.subplots(2, 3, figsize=(15, 10), facecolor='white')
        axes = axes.flatten()
        
        for idx, (variant_name, results) in enumerate(all_results.items()):
            ax = axes[idx]
            ax.set_facecolor('white')
            payoffs = results['final_expected_payoffs']
            
            n_bins = min(20, len(payoffs) // 3)
            ax.hist(payoffs, bins=n_bins, density=True, alpha=0.7,
                    color='steelblue', edgecolor='white')
            ax.axvline(x=0.875, color='red', linestyle='--', linewidth=1.5)
            ax.axvline(x=results['mean_final_expected'], color='green',
                       linestyle='-', linewidth=1.5)
            
            ax.set_title(f'{variant_name}\n'
                         f'Mean: {results["mean_final_expected"]:.3f}, '
                         f'Success: {results["success_rate"]*100:.0f}%',
                         fontsize=10, color='black')
            ax.set_xlim(0, 1.05)
            ax.set_xlabel('Final Expected Payoff', color='black')
            if idx % 3 == 0:
                ax.set_ylabel('Density', color='black')
            ax.tick_params(colors='black')
            for spine in ax.spines.values():
                spine.set_color('black')
        
        plt.suptitle(f'Comparison of All Variants\n'
                     f'n={N}, p(observe)={P_OBSERVE}, T={TIMESTEPS}, {N_RUNS} runs',
                     fontsize=12, color='black')
        plt.tight_layout()
        plt.savefig('variant_comparison.png', dpi=150, bbox_inches='tight',
                    facecolor='white', edgecolor='none')
        print("Comparison plot saved to variant_comparison.png")
        if show_plots:
            plt.show()
        else:
            plt.close()
    
    return all_results


def run_so_dual_threshold(p_observe: float = P_OBSERVE,
                           timesteps: int = TIMESTEPS,
                           n_runs: int = N_RUNS,
                           thresholds: tuple = (0.8, 0.875),
                           save_plots: bool = True,
                           show_plots: bool = False) -> dict:
    """
    Run SO, no-gating, n=4 and evaluate success rates at two thresholds simultaneously.

    Returns the full results dict with an added 'threshold_stats' key:
        results['threshold_stats'][thresh] = {
            'success_rate': float,   # fraction of runs >= thresh
            'n_success':   int,
            'n_fail':      int,
        }
    """
    config = GameConfig(
        n=4,
        p_observe=p_observe,
        action_selection=ActionSelectionEnum.SO,
        reinforcement=ReinforcementEnum.NO_GATING,
        timesteps=timesteps,
        n_runs=n_runs,
    )

    print(f"\n{'='*60}")
    print(f"Dual-Threshold Analysis: SO, no_gating, n=4")
    print(f"  p(observe)={p_observe}, T={timesteps}, runs={n_runs}")
    print(f"  Thresholds: {thresholds}")
    print(f"{'='*60}\n")

    results = run_experiments(config, verbose=True)
    payoffs = results['final_expected_payoffs']

    # Evaluate both thresholds
    threshold_stats = {}
    for thresh in thresholds:
        rate = float(np.mean(payoffs >= thresh))
        threshold_stats[thresh] = {
            'success_rate': rate,
            'n_success': int(np.sum(payoffs >= thresh)),
            'n_fail':    int(np.sum(payoffs < thresh)),
        }

    # Print results table
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print(f"\n[{timestamp}] Dual-Threshold Results (SO, no_gating, n=4):")
    print(f"  Mean final expected payoff : {results['mean_final_expected']:.4f}")
    print(f"  Mean average payoff        : {results['mean_average_payoff']:.4f}")
    print(f"  {'Threshold':<12} {'Success %':>10}  {'Pass/Total':>12}")
    print(f"  {'-'*38}")
    for thresh, stats in threshold_stats.items():
        print(f"  ≥ {thresh:<10} {stats['success_rate']*100:>9.1f}%  "
              f"{stats['n_success']:>5}/{n_runs}")

    # Plot
    if save_plots or show_plots:
        plt.style.use('default')
        fig, ax = plt.subplots(figsize=(10, 6), facecolor='white')
        ax.set_facecolor('white')

        n_bins = min(30, len(payoffs) // 3)
        ax.hist(payoffs, bins=n_bins, density=True, alpha=0.7,
                color='steelblue', edgecolor='white', linewidth=0.5,
                label='Payoff distribution')

        threshold_colors = ['red', 'darkorange']
        for (thresh, stats), color in zip(threshold_stats.items(), threshold_colors):
            ax.axvline(x=thresh, color=color, linestyle='--', linewidth=2,
                       label=f'Threshold {thresh}: {stats["success_rate"]*100:.1f}% success')

        mean_payoff = results['mean_final_expected']
        ax.axvline(x=mean_payoff, color='green', linestyle='-', linewidth=2,
                   label=f'Mean = {mean_payoff:.3f}')

        ax.set_xlabel('Final Expected Payoff', fontsize=12, color='black')
        ax.set_ylabel('Density', fontsize=12, color='black')
        ax.set_title(
            f'SO, no_gating — Dual-Threshold Analysis\n'
            f'n=4, p(observe)={p_observe}, T={timesteps}, runs={n_runs}',
            fontsize=11, color='black'
        )
        ax.legend(loc='upper left', facecolor='white', edgecolor='black')
        ax.set_xlim(0, 1.05)
        ax.tick_params(colors='black')
        for spine in ax.spines.values():
            spine.set_color('black')

        plt.tight_layout()

        if save_plots:
            path = 'so_dual_threshold.png'
            plt.savefig(path, dpi=150, bbox_inches='tight',
                        facecolor='white', edgecolor='none')
            print(f"Plot saved to {path}", flush=True)
        if show_plots:
            plt.show()
        else:
            plt.close()

    results['threshold_stats'] = threshold_stats
    return results


def warmup():
    """Run a small simulation to trigger JIT compilation."""
    print("Warming up JIT compiler...", flush=True)
    start = time.time()
    run_multiple_simulations(2, 0.1, 0, 0, 100, 2)
    elapsed = time.time() - start
    print(f"Warmup complete ({elapsed:.2f}s)\n", flush=True)


def main():
    """Main function - uses global configuration parameters."""
    
    # Warmup JIT
    warmup()
    
    # Print current configuration
    print("="*60, flush=True)
    print("CURRENT CONFIGURATION", flush=True)
    print("="*60, flush=True)
    print(f"  n (states/signals/actions): {N}", flush=True)
    print(f"  p(observe):                 {P_OBSERVE}", flush=True)
    print(f"  timesteps:                  {TIMESTEPS}", flush=True)
    print(f"  n_runs:                     {N_RUNS}", flush=True)
    print(f"  action_selection:           {ACTION_SELECTION}", flush=True)
    print(f"  reinforcement:              {REINFORCEMENT}", flush=True)
    print(f"  run_single:                 {RUN_SINGLE}", flush=True)
    print(f"  run_comparison:             {RUN_COMPARISON}", flush=True)
    print(f"  run_dual_threshold:         {RUN_DUAL_THRESHOLD}", flush=True)
    print(f"  show_failed_strategies:     {SHOW_FAILED_STRATEGIES}", flush=True)
    print("="*60 + "\n", flush=True)
    
    # Run single variant
    if RUN_SINGLE:
        print("="*60, flush=True)
        print(f"Running single variant: {ACTION_SELECTION}, {REINFORCEMENT}", flush=True)
        print("="*60, flush=True)
        
        config = GameConfig.from_globals()
        results = run_experiments(config, verbose=True)
        
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        print(f"\n[{timestamp}] Results for {config.variant_name()}:", flush=True)
        print(f"  Mean final expected payoff: {results['mean_final_expected']:.4f}", flush=True)
        print(f"  Mean average payoff: {results['mean_average_payoff']:.4f}", flush=True)
        print(f"  Success rate (≥0.875): {results['success_rate']*100:.1f}%", flush=True)
        # Dual-threshold summary
        payoffs = results['final_expected_payoffs']
        for thresh in (0.8, 0.875):
            n_success = int(np.sum(payoffs >= thresh))
            rate = n_success / config.n_runs * 100
            print(f"  Success rate (≥{thresh}): {rate:.1f}%  ({n_success}/{config.n_runs})", flush=True)
        
        
        if SAVE_PLOTS:
            plot_payoff_distribution(results, config,
                                      save_path='single_variant_dist.png',
                                      show=SHOW_PLOTS)
        print()
    
    # Run comparison of all variants
    if RUN_COMPARISON:
        all_results = compare_variants(save_plots=SAVE_PLOTS, show_plots=SHOW_PLOTS)
        
        # Print summary table
        print("\n" + "="*60)
        print("SUMMARY TABLE")
        print("="*60)
        print(f"{'Variant':<20} {'Mean Exp.':<12} {'Mean Avg.':<12} {'Success %':<10}")
        print("-"*54)
        for variant_name, results in all_results.items():
            print(f"{variant_name:<20} {results['mean_final_expected']:<12.4f} "
                  f"{results['mean_average_payoff']:<12.4f} {results['success_rate']*100:<10.1f}")

    # Run dual-threshold analysis
    if RUN_DUAL_THRESHOLD:
        run_so_dual_threshold(
            p_observe=P_OBSERVE,
            timesteps=TIMESTEPS,
            n_runs=N_RUNS,
            thresholds=(0.8, 0.875),
            save_plots=SAVE_PLOTS,
            show_plots=SHOW_PLOTS,
        )
        
    if RUN_SCALING_TABLE:
        run_scaling_table(n=N, save_dir="scaling_results")


import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import csv
import os
import time
from datetime import datetime


def run_scaling_table(
    n: int = 4,
    timestep_grid=None,
    p_observe_grid=None,
    n_runs_default: int = 1_000,
    n_runs_large: int = 100,
    large_timestep_threshold: int = 10_000_000,
    success_threshold: float = 0.875,
    save_dir: str = ".",
    save_csv: bool = True,
    save_heatmap: bool = True,
    show_plots: bool = False,
    verbose: bool = True,
):
    """
    Run all 6 variants × |timestep_grid| × |p_observe_grid| cells and
    produce per-variant tables (CSV + heatmap PNG) plus a summary CSV.
    Also saves per-cell payoff distribution plots and exports all 6 tables
    to a combined human-readable text file.

    Returns
    -------
    dict  { variant_name -> dict with keys:
              'success_rate_table'  : 2-D np.ndarray (timesteps × p_observe)
              'avg_payoff_table'    : 2-D np.ndarray (timesteps × p_observe)
          }
    """

    # ------------------------------------------------------------------ grid
    if timestep_grid is None:
        timestep_grid = [100, 1_000, 10_000, 100_000, 1_000_000, 10_000_000]
    if p_observe_grid is None:
        p_observe_grid = [0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0]

    timestep_grid = list(timestep_grid)
    p_observe_grid = list(p_observe_grid)

    os.makedirs(save_dir, exist_ok=True)

    # Directory for per-cell distribution plots
    dist_dir = os.path.join(save_dir, "payoff_distributions")
    os.makedirs(dist_dir, exist_ok=True)

    # --------------------------------------------------------- variant specs
    variants = [
        ("LP",  "no_gating", int(ActionSelectionEnum.LP),  int(ReinforcementEnum.NO_GATING)),
        ("LP",  "gated",     int(ActionSelectionEnum.LP),  int(ReinforcementEnum.GATED)),
        ("BP",  "no_gating", int(ActionSelectionEnum.BP),  int(ReinforcementEnum.NO_GATING)),
        ("BP",  "gated",     int(ActionSelectionEnum.BP),  int(ReinforcementEnum.GATED)),
        ("SO",  "no_gating", int(ActionSelectionEnum.SO),  int(ReinforcementEnum.NO_GATING)),
        ("SO",  "gated",     int(ActionSelectionEnum.SO),  int(ReinforcementEnum.GATED)),
    ]

    # all_tables[variant_name] = {'success_rate_table': ..., 'avg_payoff_table': ...}
    all_tables = {}

    grand_start = time.time()

    for action_label, reinf_label, action_sel, reinf in variants:
        variant_name = f"{action_label}, {reinf_label}"

        if verbose:
            ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            print(f"\n[{ts}] ═══ Variant: {variant_name} ═══", flush=True)

        success_table = np.full((len(timestep_grid), len(p_observe_grid)), np.nan)
        avg_payoff_table = np.full((len(timestep_grid), len(p_observe_grid)), np.nan)

        for row_idx, timesteps in enumerate(timestep_grid):
            n_runs = (
                n_runs_large if timesteps >= large_timestep_threshold
                else n_runs_default
            )

            for col_idx, p_obs in enumerate(p_observe_grid):
                cell_start = time.time()

                # ---- run simulations ----
                final_payoffs, avg_payoffs = run_multiple_simulations(
                    n,
                    p_obs,
                    action_sel,
                    reinf,
                    timesteps,
                    n_runs,
                )

                success_rate = float(np.mean(final_payoffs >= success_threshold))
                mean_avg_payoff = float(np.mean(avg_payoffs))

                success_table[row_idx, col_idx] = success_rate
                avg_payoff_table[row_idx, col_idx] = mean_avg_payoff

                # ---- save per-cell distribution plot ----
                safe_variant = variant_name.replace(", ", "_")
                dist_fname = (
                    f"dist_{safe_variant}_T{_fmt_timestep(timesteps)}"
                    f"_p{p_obs:.1f}.png"
                )
                dist_path = os.path.join(dist_dir, dist_fname)
                _save_cell_distribution_plot(
                    final_payoffs, variant_name, n, timesteps, p_obs,
                    n_runs, success_threshold, success_rate,
                    mean_avg_payoff, dist_path
                )

                if verbose:
                    elapsed = time.time() - cell_start
                    ts = datetime.now().strftime("%H:%M:%S")
                    print(
                        f"  [{ts}] T={timesteps:>10,}  p={p_obs:.1f}  "
                        f"runs={n_runs:>5}  success={success_rate*100:5.1f}%"
                        f"  avg_payoff={mean_avg_payoff:.4f}"
                        f"  ({elapsed:.1f}s)",
                        flush=True,
                    )

        all_tables[variant_name] = {
            'success_rate_table': success_table,
            'avg_payoff_table': avg_payoff_table,
        }

        # -------------------------------------------------- print ascii table
        if verbose:
            _print_ascii_table(
                success_table, avg_payoff_table,
                timestep_grid, p_observe_grid, variant_name, success_threshold
            )

        # ----------------------------------------------------------- save CSV
        if save_csv:
            safe = variant_name.replace(", ", "_")
            csv_path = os.path.join(save_dir, f"scaling_{safe}.csv")
            _write_variant_csv(
                csv_path, success_table, avg_payoff_table,
                timestep_grid, p_observe_grid,
                variant_name, n_runs_default, n_runs_large,
                large_timestep_threshold, success_threshold
            )
            if verbose:
                print(f"  CSV  → {csv_path}", flush=True)

        # --------------------------------------------------------- heatmap
        if save_heatmap or show_plots:
            safe = variant_name.replace(", ", "_")
            png_path = os.path.join(save_dir, f"scaling_heatmap_{safe}.png") if save_heatmap else None
            _plot_heatmap(
                success_table, avg_payoff_table,
                timestep_grid, p_observe_grid,
                variant_name, n, success_threshold,
                n_runs_default, n_runs_large, large_timestep_threshold,
                save_path=png_path, show=show_plots,
            )
            if verbose and png_path:
                print(f"  PNG  → {png_path}", flush=True)

    # --------------------------------------------------- summary CSV (all 6)
    if save_csv:
        summary_path = os.path.join(save_dir, "scaling_summary_all_variants.csv")
        _write_summary_csv(
            summary_path, all_tables, timestep_grid, p_observe_grid,
            success_threshold
        )
        if verbose:
            print(f"\nSummary CSV → {summary_path}", flush=True)

    # ---------------------------------------- export all 6 tables to one text file
    tables_txt_path = os.path.join(save_dir, "all_tables.txt")
    _write_all_tables_txt(
        tables_txt_path, all_tables, timestep_grid, p_observe_grid,
        success_threshold, n, n_runs_default, n_runs_large, large_timestep_threshold
    )
    if verbose:
        print(f"All tables (text) → {tables_txt_path}", flush=True)

    total_elapsed = time.time() - grand_start
    if verbose:
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        print(f"\n[{ts}] All variants complete. Total time: {total_elapsed:.1f}s", flush=True)
        print(f"Distribution plots saved in: {dist_dir}/", flush=True)

    return all_tables


# =============================================================================
# Helper functions
# =============================================================================

def _fmt_timestep(t: int) -> str:
    """Pretty-print large integers as e.g. '1e4', '1e7'."""
    exponent = round(np.log10(t))
    if 10 ** exponent == t:
        return f"1e{exponent}"
    return f"{t:,}"


def _print_ascii_table(success_table, avg_payoff_table,
                        timestep_grid, p_observe_grid, variant_name, threshold):
    """Print a combined success-rate + avg-payoff table to stdout."""
    col_w = 14  # wider to fit "XX.X% / 0.XXX"
    header_p = "  ".join(f"p={p:.1f}".rjust(col_w) for p in p_observe_grid)
    sep = "─" * (14 + len(header_p))
    print(f"\n  {variant_name}  (success ≥ {threshold})", flush=True)
    print(f"  Format: success% / avg_payoff", flush=True)
    print(f"  {'plays/run':<14}{header_p}", flush=True)
    print(f"  {sep}", flush=True)
    for row_idx, t in enumerate(timestep_grid):
        row_str = "  ".join(
            f"{success_table[row_idx, c]*100:.1f}%/{avg_payoff_table[row_idx, c]:.3f}".rjust(col_w)
            for c in range(len(p_observe_grid))
        )
        print(f"  {_fmt_timestep(t):<14}{row_str}", flush=True)
    print(flush=True)


def _write_variant_csv(path, success_table, avg_payoff_table,
                        timestep_grid, p_observe_grid,
                        variant_name, n_runs_default, n_runs_large,
                        large_threshold, threshold):
    """Write a single variant's table to CSV with both success rate and avg payoff."""
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        # metadata rows
        w.writerow(["# variant", variant_name])
        w.writerow(["# success_threshold", threshold])
        w.writerow(["# n_runs_default", n_runs_default])
        w.writerow([f"# n_runs_for_T>={large_threshold}", n_runs_large])
        w.writerow([])
        # --- success rate block ---
        w.writerow(["## success_rate"])
        w.writerow(["plays_per_run"] + [f"p={p}" for p in p_observe_grid])
        for row_idx, t in enumerate(timestep_grid):
            w.writerow(
                [_fmt_timestep(t)]
                + [f"{success_table[row_idx, c]:.4f}" for c in range(len(p_observe_grid))]
            )
        w.writerow([])
        # --- avg payoff block ---
        w.writerow(["## avg_payoff"])
        w.writerow(["plays_per_run"] + [f"p={p}" for p in p_observe_grid])
        for row_idx, t in enumerate(timestep_grid):
            w.writerow(
                [_fmt_timestep(t)]
                + [f"{avg_payoff_table[row_idx, c]:.4f}" for c in range(len(p_observe_grid))]
            )


def _write_summary_csv(path, all_tables, timestep_grid, p_observe_grid, threshold):
    """
    Wide-format summary: rows = (variant, timestep), cols = p_observe values.
    Includes both success rate and avg payoff blocks per variant.
    """
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["# Scaling table — all 6 variants"])
        w.writerow([f"# success_threshold = {threshold}"])
        w.writerow([])

        for variant_name, tables in all_tables.items():
            success_table = tables['success_rate_table']
            avg_table = tables['avg_payoff_table']

            w.writerow([f"### {variant_name}"])

            w.writerow(["## success_rate"])
            w.writerow(["plays_per_run"] + [f"p={p}" for p in p_observe_grid])
            for row_idx, t in enumerate(timestep_grid):
                w.writerow(
                    [_fmt_timestep(t)]
                    + [f"{success_table[row_idx, c]:.4f}" for c in range(len(p_observe_grid))]
                )

            w.writerow(["## avg_payoff"])
            w.writerow(["plays_per_run"] + [f"p={p}" for p in p_observe_grid])
            for row_idx, t in enumerate(timestep_grid):
                w.writerow(
                    [_fmt_timestep(t)]
                    + [f"{avg_table[row_idx, c]:.4f}" for c in range(len(p_observe_grid))]
                )

            w.writerow([])  # blank line between variants


def _write_all_tables_txt(path, all_tables, timestep_grid, p_observe_grid,
                           threshold, n, n_runs_default, n_runs_large,
                           large_threshold):
    """
    Export all 6 variant tables (both success rate and avg payoff) to a
    single human-readable text file.
    """
    col_w = 14

    with open(path, "w") as f:
        f.write("=" * 80 + "\n")
        f.write("SCALING TABLE RESULTS — ALL 6 VARIANTS\n")
        f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"n={n}, success_threshold={threshold}\n")
        f.write(f"n_runs: {n_runs_default} (T<{large_threshold}), "
                f"{n_runs_large} (T>={large_threshold})\n")
        f.write("Format per cell: success% / avg_payoff\n")
        f.write("=" * 80 + "\n\n")

        for variant_name, tables in all_tables.items():
            success_table = tables['success_rate_table']
            avg_table = tables['avg_payoff_table']

            f.write("─" * 80 + "\n")
            f.write(f"Variant: {variant_name}\n")
            f.write("─" * 80 + "\n\n")

            # Combined table header
            header_p = "  ".join(f"p={p:.1f}".rjust(col_w) for p in p_observe_grid)
            sep = "─" * (14 + len(header_p))
            f.write(f"  {'plays/run':<14}{header_p}\n")
            f.write(f"  {sep}\n")
            for row_idx, t in enumerate(timestep_grid):
                row_str = "  ".join(
                    f"{success_table[row_idx, c]*100:.1f}%/{avg_table[row_idx, c]:.3f}".rjust(col_w)
                    for c in range(len(p_observe_grid))
                )
                f.write(f"  {_fmt_timestep(t):<14}{row_str}\n")

            f.write("\n")

            # Success rate only sub-table
            f.write("  Success rate (≥{}) only:\n".format(threshold))
            f.write(f"  {'plays/run':<14}" +
                    "  ".join(f"p={p:.1f}".rjust(8) for p in p_observe_grid) + "\n")
            f.write("  " + "─" * (14 + 10 * len(p_observe_grid)) + "\n")
            for row_idx, t in enumerate(timestep_grid):
                row_str = "  ".join(
                    f"{success_table[row_idx, c]*100:5.1f}%".rjust(8)
                    for c in range(len(p_observe_grid))
                )
                f.write(f"  {_fmt_timestep(t):<14}{row_str}\n")

            f.write("\n")

            # Avg payoff only sub-table
            f.write("  Average payoff only:\n")
            f.write(f"  {'plays/run':<14}" +
                    "  ".join(f"p={p:.1f}".rjust(8) for p in p_observe_grid) + "\n")
            f.write("  " + "─" * (14 + 10 * len(p_observe_grid)) + "\n")
            for row_idx, t in enumerate(timestep_grid):
                row_str = "  ".join(
                    f"{avg_table[row_idx, c]:.4f}".rjust(8)
                    for c in range(len(p_observe_grid))
                )
                f.write(f"  {_fmt_timestep(t):<14}{row_str}\n")

            f.write("\n\n")

        f.write("=" * 80 + "\n")
        f.write("END OF REPORT\n")
        f.write("=" * 80 + "\n")


def _save_cell_distribution_plot(final_payoffs, variant_name, n, timesteps,
                                   p_obs, n_runs, success_threshold,
                                   success_rate, mean_avg_payoff, save_path):
    """Save a payoff distribution histogram for a single (variant, T, p_obs) cell."""
    plt.style.use("default")
    fig, ax = plt.subplots(figsize=(8, 5), facecolor="white")
    ax.set_facecolor("white")

    n_bins = min(30, max(5, n_runs // 10))
    ax.hist(final_payoffs, bins=n_bins, density=True, alpha=0.7,
            color="steelblue", edgecolor="white", linewidth=0.5)

    ax.axvline(x=success_threshold, color="red", linestyle="--", linewidth=1.5,
               label=f"Threshold {success_threshold}: {success_rate*100:.1f}%")
    mean_ep = float(np.mean(final_payoffs))
    ax.axvline(x=mean_ep, color="green", linestyle="-", linewidth=1.5,
               label=f"Mean exp. payoff = {mean_ep:.3f}")
    ax.axvline(x=mean_avg_payoff, color="orange", linestyle=":", linewidth=1.5,
               label=f"Mean avg payoff = {mean_avg_payoff:.3f}")

    ax.set_xlabel("Final Expected Payoff", fontsize=10, color="black")
    ax.set_ylabel("Density", fontsize=10, color="black")
    ax.set_title(
        f"{variant_name} | n={n}, T={timesteps:,}, p={p_obs:.1f}, runs={n_runs}\n"
        f"Success: {success_rate*100:.1f}%  |  Avg payoff: {mean_avg_payoff:.4f}",
        fontsize=9, color="black"
    )
    ax.legend(fontsize=8, facecolor="white", edgecolor="black")
    ax.set_xlim(0, 1.05)
    ax.tick_params(colors="black")
    for spine in ax.spines.values():
        spine.set_color("black")

    plt.tight_layout()
    plt.savefig(save_path, dpi=120, bbox_inches="tight",
                facecolor="white", edgecolor="none")
    plt.close()


def _plot_heatmap(success_table, avg_payoff_table,
                   timestep_grid, p_observe_grid,
                   variant_name, n, threshold,
                   n_runs_default, n_runs_large, large_threshold,
                   save_path=None, show=False):
    """
    Produce a color-coded heatmap of success rates for one variant.
    Each cell shows both success % and average payoff.

    Color scale: white (0 %) → deep teal (100 %)
    """
    plt.style.use("default")

    nrows, ncols = len(timestep_grid), len(p_observe_grid)
    fig_w = max(10, ncols * 1.4 + 2)
    fig_h = max(5, nrows * 0.85 + 2)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h), facecolor="white")
    ax.set_facecolor("white")

    # Custom colormap: white → teal
    cmap = mcolors.LinearSegmentedColormap.from_list(
        "wt", ["#ffffff", "#0e6655", "#1abc9c"]
    )

    im = ax.imshow(success_table * 100, aspect="auto", cmap=cmap,
                   vmin=0, vmax=100, interpolation="nearest")

    # Annotate cells with success% and avg payoff
    for r in range(nrows):
        for c in range(ncols):
            s_val = success_table[r, c] * 100
            a_val = avg_payoff_table[r, c]
            txt_color = "black" if s_val < 60 else "white"
            n_runs = n_runs_large if timestep_grid[r] >= large_threshold else n_runs_default
            ax.text(c, r,
                    f"{s_val:.0f}%\navg:{a_val:.3f}\n(n={n_runs})",
                    ha="center", va="center", fontsize=7.5,
                    color=txt_color, fontweight="bold")

    # Axis labels
    ax.set_xticks(range(ncols))
    ax.set_xticklabels([f"p={p}" for p in p_observe_grid], fontsize=9)
    ax.set_yticks(range(nrows))
    ax.set_yticklabels([_fmt_timestep(t) for t in timestep_grid], fontsize=9)
    ax.set_xlabel("p(observe)", fontsize=11)
    ax.set_ylabel("Plays per run", fontsize=11)

    title = (
        f"Success rate (≥{threshold}) & Avg Payoff — {variant_name}\n"
        f"n={n},  runs={n_runs_default} (1e2–1e6), {n_runs_large} (1e7)"
    )
    ax.set_title(title, fontsize=11, pad=10)

    cb = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.04)
    cb.set_label("Success rate (%)", fontsize=9)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight",
                    facecolor="white", edgecolor="none")
    if show:
        plt.show()
    else:
        plt.close()


if __name__ == "__main__":
    main()

Warming up JIT compiler...
Warmup complete (3.41s)

CURRENT CONFIGURATION
  n (states/signals/actions): 4
  p(observe):                 1
  timesteps:                  10000000
  n_runs:                     100
  action_selection:           BP
  reinforcement:              no_gating
  run_single:                 True
  run_comparison:             False
  run_dual_threshold:         False
  show_failed_strategies:     False

Running single variant: BP, no_gating
Running 100 simulations...
  [2026-04-01 23:16:30] Progress: 1/100 runs completed
  [2026-04-01 23:16:34] Progress: 2/100 runs completed
  [2026-04-01 23:16:38] Progress: 3/100 runs completed
  [2026-04-01 23:16:41] Progress: 4/100 runs completed
  [2026-04-01 23:16:45] Progress: 5/100 runs completed
  [2026-04-01 23:16:49] Progress: 6/100 runs completed
  [2026-04-01 23:16:52] Progress: 7/100 runs completed
  [2026-04-01 23:16:56] Progress: 8/100 runs completed
  [2026-04-01 23:17:00] Progress: 9/100 runs completed
  [2026-04-0